In [1]:
!pip install git+https://github.com/fangwei123456/spikingjelly.git


  Cloning https://github.com/fangwei123456/spikingjelly.git to /tmp/pip-req-build-umr1_y5x
  Running command git clone --filter=blob:none --quiet https://github.com/fangwei123456/spikingjelly.git /tmp/pip-req-build-umr1_y5x
  Resolved https://github.com/fangwei123456/spikingjelly.git to commit 075f97e91ab22dabfd29651231f9fc625aa95fbf
  Preparing metadata (setup.py) ... done
  Created wheel for spikingjelly: filename=spikingjelly-0.0.0.0.15-py3-none-any.whl size=387294 sha256=c25f12c5db803706a93c91fa67037d8627293d281a3dd2cbb90f8080a834a1b4
  Stored in directory: /tmp/pip-ephem-wheel-cache-ieexcsm9/wheels/93/ab/c9/e0318e4c9feaebc1632bcc0168638a2ed5ef17340741c9ad49
Successfully built spikingjelly


# KS Small Fully Spiking

In [2]:
import torch
import torch.nn as nn
import datetime
from tqdm import tqdm
import torch.nn.functional as F
import torchvision
import torchvision.datasets as dset
from torchvision import transforms
from typing import Union
from torch.nn.common_types import _size_2_t
from torch.utils.data import DataLoader, random_split
from spikingjelly.datasets.cifar10_dvs import CIFAR10DVS
from spikingjelly.activation_based import base, functional, layer, surrogate, neuron, learning, monitor#, accelerating
from spikingjelly.activation_based.neuron import ParametricLIFNode as PLIFNode
from spikingjelly.activation_based.neuron import GatedLIFNode as GLIFNode
from spikingjelly.activation_based.neuron import LIFNode
import numpy as np

import pandas as pd
import os
import h5py
import argparse
import time
import sys
import matplotlib.pyplot as plt



# from spikingjelly.activation_based.model.sew_resnet import sew_resnet18
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
import sys
sys.path.append('/content/drive/MyDrive/FORTH/Colab Codes')
from helpers import log_tau_after_epoch
from spiking_resnet_archs import *

In [4]:
import cupy

## Set-up

In [5]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [6]:
torch.backends.cudnn.benchmark = True
_seed_ = 419
torch.manual_seed(_seed_)  # use torch.manual_seed() to seed the RNG for all devices (both CPU and CUDA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(_seed_)

In [7]:
# from spiking_resnet_archs import resnet18, resnet34, sew_resnet18
device = 'cuda:0'
step_mode = 'm'
init_tau = 2.0
detach_reset = True
class_num = 10
cnf = 'ADD'
surrogate_function = surrogate.ATan()
bn_momentum = 0.1

In [8]:
# net = sew_resnet18(cnf = 'ADD', spiking_neuron=PLIFNode).to(device)
net = KS32_FullySpiking_Small(BasicBlock, [1,1,1], num_classes = class_num, bn_momentum=bn_momentum,
                        cnf=cnf , spiking_neuron=PLIFNode, init_tau=init_tau, detach_reset=detach_reset, surrogate_function=surrogate_function).to(device)
functional.set_step_mode(net, step_mode=step_mode)
functional.set_backend(net, backend='cupy')
net

KS32_FullySpiking_Small(
  (layer1): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=m)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
    (2): ParametricLIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=cupy, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
  )
  (layer2): Sequential(
    (0): BasicBlock(
      cnf=ADD
      (downsample): Sequential()
      (residual_function): Sequential(
        (0): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=m)
        (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
        (2): ParametricLIFNode(
          v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=cupy, tau=2.0
          (surrogate_function): ATan(alpha=2.0, spiking=True)
        )
        (3): Conv2d(8, 8, kern

In [9]:
def cal_firing_rate(s_seq: torch.Tensor):
    return s_seq.mean()

In [10]:
fr_monitor = monitor.OutputMonitor(net, PLIFNode, cal_firing_rate)

In [13]:

batch_size = 250
learning_rate = 1e-3
T_max = 200

T = 8


num_workers = 1

exc_ratio = 0.75

tau_pre = 2.
tau_post = 2.
feedback_gain = 0.4


In [ ]:
# cifar100_dir = '/content/drive/MyDrive/FORTH/Colab Codes/CIFAR10'
# dataset=torchvision.datasets.CIFAR10(root=cifar100_dir, train=True, download=True)

In [ ]:
cifar100_dir = '/content/drive/MyDrive/FORTH/Colab Codes/CIFAR10'
train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.05)),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

norm_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

train_data_loader = torch.utils.data.DataLoader(
            dataset=torchvision.datasets.CIFAR10(root=cifar100_dir, train=True, transform=train_transform, download=False),
            batch_size=batch_size,
            shuffle=True,
            num_workers= num_workers,
            drop_last=True,
            pin_memory=True)

test_data_loader = torch.utils.data.DataLoader(
            dataset=torchvision.datasets.CIFAR10(root=cifar100_dir, train=False, transform=norm_transform, download=False),
            batch_size=batch_size//2,
            shuffle=True,
            num_workers= num_workers,
            drop_last=True,
            pin_memory=True)

print('!!!!!! RUNNING CIFAR10 !!!!!!!!!!!')

!!!!!! RUNNING CIFAR10 !!!!!!!!!!!


In [ ]:
# img, lab = next(iter(train_data_loader))

In [ ]:
# out = net(img.to(device))
# out = net(img.unsqueeze(0).repeat(T,1,1,1,1).to(device))

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max)

In [ ]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [ ]:
device

'cuda:0'

## Train

In [ ]:
tau_tracker = {}
tracker = pd.DataFrame(columns = ['train_acc', 'train_loss', 'train_speed', 'test_acc', 'test_acc3', 'test_acc5', 'test_loss', 'test_speed', 'epoch_time', 'lr'])

In [ ]:
# max_rate = 0.1
# lambda_rate = 0.01

net.train_times = 0
max_test_acc = -1
start_epoch = 0
max_epoch = 300

start_epoch, max_epoch

(0, 300)

In [ ]:
out_dir = f'/content/drive/MyDrive/FORTH/Colab Codes/models/C10_T{T}_[KS32_FullSNN_Small]_bn[{bn_momentum}]_tau[{init_tau}]_batch[{batch_size}]_cnf[{cnf}]'
out_dir

'/content/drive/MyDrive/FORTH/Colab Codes/models/C10_T8_[KS32_FullSNN_Small]_bn[0.1]_tau[2.0]_batch[250]_cnf[ADD]'

In [ ]:

if not os.path.exists(out_dir):
    os.mkdir(out_dir)
    print('y')
checkpoint_path = os.path.join(out_dir, 'checkpoint_latest.pth')

y


In [ ]:
for epoch in range(start_epoch, max_epoch):

    start_time = datetime.datetime.now()
    print(f'<> Epoch :  {epoch}, {(datetime.datetime.now().strftime("%H:%M:%S"))}')

    net.train()
    fr_monitor.enable()
    train_loss = 0
    train_acc = 0
    train_samples = 0

    train_start = datetime.datetime.now()
    for img, label in tqdm(train_data_loader, desc=f"Training Epoch {epoch}", leave=False):
        img = img.unsqueeze(0)
        img = img.repeat(T,1,1,1,1)
        img = img.to(device)
        label = label.to(device)
        optimizer.zero_grad()

        out_spikes_counter = net(img)
        out_spikes_counter_frequency = out_spikes_counter.mean(axis=0)

        loss = F.cross_entropy(out_spikes_counter_frequency, F.one_hot(label, class_num).float())

        # firing_rate_reg = firing_rate_reg = sum(torch.relu(r - max_rate) for r in fr_monitor.records)
        # loss += lambda_rate * firing_rate_reg

        loss.backward()
        optimizer.step()
        functional.reset_net(net)
        fr_monitor.clear_recorded_data()

        train_samples += 1
        train_loss += loss.item()
        train_acc += (out_spikes_counter_frequency.argmax(dim=1) == label).float().mean().item()

        del out_spikes_counter, out_spikes_counter_frequency


    train_stop = datetime.datetime.now()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    net.train_times += 1
    train_speed = train_samples / (train_stop - train_start).total_seconds()
    train_loss /= train_samples
    train_acc /= train_samples
    scheduler.step()

    net.eval()
    fr_monitor.disable()
    log_tau_after_epoch(net, tau_tracker)
    test_start = datetime.datetime.now()
    with torch.no_grad():
        test_loss = 0
        test_acc = 0
        test_acc3 = 0
        test_acc5 = 0
        test_samples = 0
        for img, label in tqdm(test_data_loader, desc=f"Testing Epoch {epoch}", leave=False):
            img = img.unsqueeze(0)
            img = img.repeat(T,1,1,1,1)
            img = img.to(device)
            label = label.to(device)
            out_spikes_counter = net(img)
            out_spikes_counter_frequency = out_spikes_counter.mean(axis=0)
            loss = F.cross_entropy(out_spikes_counter_frequency, F.one_hot(label, class_num).float())
            test_loss += loss.item()
            test_samples += 1

            # Acc measures
            top1 = out_spikes_counter_frequency.topk(1, dim=1).indices
            top3 = out_spikes_counter_frequency.topk(3, dim=1).indices
            top5 = out_spikes_counter_frequency.topk(5, dim=1).indices

            top1_correct = (top1 == label.unsqueeze(1)).any(dim=1)
            top3_correct = (top3 == label.unsqueeze(1)).any(dim=1)
            top5_correct = (top5 == label.unsqueeze(1)).any(dim=1)

            acc1 = top1_correct.float().mean().item()
            acc3 = top3_correct.float().mean().item()
            acc5 = top5_correct.float().mean().item()

            test_acc += acc1
            test_acc3 += acc3
            test_acc5 += acc5

            #reset

            functional.reset_net(net)
            del out_spikes_counter, out_spikes_counter_frequency


        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        test_stop = datetime.datetime.now()
        test_speed = test_samples / (test_start - test_stop).total_seconds()
        test_acc /= test_samples
        test_acc3 /= test_samples
        test_acc5 /= test_samples
        test_loss /= test_samples

    print()



    save_max = False
    if test_acc > max_test_acc:
        max_test_acc = test_acc
        save_max = True

    checkpoint = {
        'net': net.state_dict(),
        'optimizer': optimizer.state_dict(),
        'lr_scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'max_test_acc': max_test_acc
    }

    tracker.loc[epoch] = [train_acc, train_loss, train_speed, test_acc, test_acc3, test_acc5, test_loss, test_speed, str(datetime.datetime.now() - start_time), optimizer.param_groups[0]['lr']]
    if save_max:
        torch.save(checkpoint, os.path.join(out_dir, 'checkpoint_max.pth'))

    torch.save(checkpoint, os.path.join(out_dir, 'checkpoint_latest.pth'))

    if np.mod(epoch, 10) == 0:
        tracker.to_csv(os.path.join(out_dir, 'tracker.csv'))
        pd.DataFrame(tau_tracker).to_csv(os.path.join(out_dir, 'tau_tracker.csv'))

    print(f'{epoch} : train_acc ={train_acc: .4f}, test_acc ={test_acc: .4f}, test_acc3 ={test_acc3: .4f}, test_acc5 ={test_acc5: .4f}, max_test_acc ={max_test_acc: .4f}')
    print(f'==> epoch time = {str(datetime.datetime.now() - start_time)}\n')

tracker.to_csv(os.path.join(out_dir, 'tracker.csv'))

<> Epoch :  0, 14:17:14



0 : train_acc = 0.2212, test_acc = 0.3497, test_acc3 = 0.6966, test_acc5 = 0.8786, max_test_acc = 0.3497
==> epoch time = 0:01:22.728497

<> Epoch :  1, 14:18:37



1 : train_acc = 0.3870, test_acc = 0.4196, test_acc3 = 0.7703, test_acc5 = 0.9143, max_test_acc = 0.4196
==> epoch time = 0:01:17.452457

<> Epoch :  2, 14:19:55



2 : train_acc = 0.4490, test_acc = 0.4905, test_acc3 = 0.8141, test_acc5 = 0.9311, max_test_acc = 0.4905
==> epoch time = 0:01:18.695243

<> Epoch :  3, 14:21:13



3 : train_acc = 0.4793, test_acc = 0.5067, test_acc3 = 0.8286, test_acc5 = 0.9373, max_test_acc = 0.5067
==> epoch time = 0:01:19.304645

<> Epoch :  4, 14:22:33


...


42 : train_acc = 0.7527, test_acc = 0.7327, test_acc3 = 0.9361, test_acc5 = 0.9821, max_test_acc = 0.7450
==> epoch time = 0:01:18.718797

<> Epoch :  43, 15:13:13



43 : train_acc = 0.7540, test_acc = 0.7440, test_acc3 = 0.9422, test_acc5 = 0.9828, max_test_acc = 0.7450
==> epoch time = 0:01:18.529947

<> Epoch :  44, 15:14:31



44 : train_acc = 0.7601, test_acc = 0.7487, test_acc3 = 0.9386, test_acc5 = 0.9843, max_test_acc = 0.7487
==> epoch time = 0:01:17.688632

<> Epoch :  45, 15:15:49



45 : train_acc = 0.7587, test_acc = 0.7482, test_acc3 = 0.9403, test_acc5 = 0.9848, max_test_acc = 0.7487
==> epoch time = 0:01:18.309145

<> Epoch :  46, 15:17:07


KeyboardInterrupt: 

In [ ]:
tracker.to_csv(os.path.join(out_dir, 'tracker.csv'))
pd.DataFrame(tau_tracker).to_csv(os.path.join(out_dir, 'tau_tracker.csv'))

In [ ]:
tracker

,train_acc,train_loss,train_speed,test_acc,test_acc3,test_acc5,test_loss,test_speed,epoch_time,lr
0,0.03694,4.336679,2.854496,0.0776,0.1800,0.2601,4.026051,-23.012914,0:01:13.581483,0.001000
1,0.09998,3.847058,3.011173,0.1310,0.2722,0.3718,3.664305,-22.440714,0:01:10.023141,0.001000
2,0.14104,3.593915,2.937790,0.1711,0.3374,0.4406,3.426676,-17.766983,0:01:12.627055,0.000999
3,0.17232,3.411971,2.931056,0.1921,0.3660,0.4665,3.326641,-23.644917,0:01:11.657463,0.000999
4,0.19738,3.274147,2.979272,0.2193,0.4073,0.5132,3.162552,-23.662576,0:01:10.551268,0.000998
...,...,...,...,...,...,...,...,...,...,...
66,0.65924,1.183130,2.958114,0.4177,0.6258,0.7143,2.415512,-22.776955,0:01:11.164513,0.000748
67,0.66514,1.160260,2.957202,0.4179,0.6245,0.7120,2.439243,-22.094119,0:01:11.293160,0.000741
68,0.66570,1.154507,2.919372,0.4241,0.6290,0.7138,2.421013,-17.252478,0:01:13.184955,0.000734
69,0.67172,1.138466,2.906506,0.4237,0.6288,0.7140,2.431066,-22.419583,0:01:12.419787,0.000727


In [ ]:
net.eval()
fr_monitor.clear_recorded_data()
fr_monitor.enable()
img, label = next(iter(test_data_loader))
img = img.unsqueeze(0)
img = img.repeat(T,1,1,1,1)
img = img.to(device)
label = label.to(device)
out_spikes_counter = net(img)
out_spikes_counter_frequency = out_spikes_counter.mean(axis=0)

In [ ]:
(out_spikes_counter_frequency.argmax(dim=1) == label).float().mean().item()

0.3440000116825104

In [ ]:
fr_monitor.records

[tensor(0.2442, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.2345, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.1924, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.2428, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.2969, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.2048, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.2297, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.1956, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.1128, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.1278, device='cuda:0', grad_fn=<MeanBackward0>),
 tensor(0.1751, device='cuda:0', grad_fn=<MeanBackward0>)]

In [ ]:
img.shape

torch.Size([8, 125, 3, 32, 32])

In [ ]:
net.layer1

Sequential(
  (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=m)
  (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
  (2): ParametricLIFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=cupy, tau=1.8965550661087036
    (surrogate_function): ATan(alpha=2.0, spiking=True)
  )
)

In [ ]:
net.layer1[0].weight.shape

torch.Size([8, 3, 3, 3])

In [ ]:
activations = {}
gradients = {}

def save_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

def save_gradient(name):
    def hook(model, grad_input, grad_output):
        gradients[name] = grad_output[0].detach()
    return hook

# Register hooks for all Conv + LIF layers
for idx, layer in enumerate(net):
    if isinstance(layer, torch.nn.Conv2d) or 'LIF' in str(type(layer)):
        layer.register_forward_hook(save_activation(f"layer_{idx}"))
        layer.register_backward_hook(save_gradient(f"layer_{idx}"))

In [ ]:

input_img = input_img.unsqueeze(0)  # add batch dim if needed
T = 50  # number of timesteps
device = "cuda" if torch.cuda.is_available() else "cpu"
net = net.to(device)
input_img = input_img.to(device)

# Store activations and gradients


# --- Forward pass over T timesteps ---
spike_record = []
x = input_img
for t in range(T):
    out = net(x)
    spike_record.append(out)

# Convert to tensor: (T, batch, neurons, H, W) depending on layer output
spike_record = torch.stack(spike_record)  # shape: [T, 1, C_out, H, W]
# Sum over time and batch to get scalar per output neuron
y = spike_record.sum(dim=(0, 1))  # shape: [C_out, H, W] or [C_out] depending on flatten

# --- Choose target neuron ---
target_neuron = 0
if y.ndim > 0:
    y_target = y[target_neuron]
else:
    y_target = y

# --- Backward pass ---
net.zero_grad()
y_target.backward(retain_graph=True)

# --- Compute Grad-CAM for each layer ---
def compute_gradcam(layer_name):
    act = activations[layer_name]        # [T, 1, C, H, W] or [1, C, H, W]
    grad = gradients[layer_name]         # same shape
    # Average gradient over channels, timesteps, batch
    weights = grad.mean(dim=(0, 2, 3))  # shape: [C]
    # Weighted sum of feature maps
    cam = torch.zeros(act.shape[2:], device=act.device)  # H x W
    for i, w in enumerate(weights):
        cam += w * act[0, i]  # take first timestep if needed
    cam = F.relu(cam)
    cam = cam / cam.max()
    return cam.cpu().numpy()

# Plot Grad-CAM for all layers
for idx, layer in enumerate(net):
    if f"layer_{idx}" in activations:
        cam = compute_gradcam(f"layer_{idx}")
        plt.figure()
        plt.imshow(cam, cmap="jet")
        plt.title(f"Grad-CAM Layer {idx}")
        plt.axis("off")
        plt.show()


1